# Ejercicio 01 — watsonx.ai: Setup y primer llamado

Este notebook tiene **dos partes**:

- **Parte 1 (obligatoria, ~5 min):** pasos 1 a 4. Si los completás, ya cumpliste el onboarding del workshop.
- **Parte 2 (opcional, ~5 min):** pasos 5 y 6. Exploración de tokens y temperatura. Si te queda tiempo, hacelos.

**Pre-requisito:** haber corrido `00_smoke_test.ipynb` con todo en `[OK]`.

---
## PARTE 1 — Setup mínimo (obligatorio)

### Paso 1 — Cargar credenciales desde `.env`

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

WX_API_KEY    = os.getenv('WX_API_KEY', '')
WX_PROJECT_ID = os.getenv('WX_PROJECT_ID', '')
WX_URL        = os.getenv('WX_URL', 'https://us-south.ml.cloud.ibm.com')

assert WX_API_KEY and WX_PROJECT_ID, 'Faltan credenciales en .env'
print('Credenciales cargadas')
print(f'  API Key:    {WX_API_KEY[:8]}...')
print(f'  Project ID: {WX_PROJECT_ID[:8]}...')
print(f'  URL:        {WX_URL}')

Credenciales cargadas
  API Key:    ZqQTjHQn...
  Project ID: 8cef60a9...
  URL:        https://us-south.ml.cloud.ibm.com


### Paso 2 — Inicializar el cliente del SDK

El SDK maneja automáticamente el refresh del IAM token a partir de la API key. Solo necesitás pasarla una vez.

In [2]:
from ibm_watsonx_ai import APIClient, Credentials

credentials = Credentials(url=WX_URL, api_key=WX_API_KEY)
client = APIClient(credentials)

print('Cliente inicializado')
print(f'SDK version: {client.version}')

Cliente inicializado
SDK version: 1.5.6


### Paso 3 — Verificar acceso al proyecto

Si este paso falla con 403, no tenés permisos sobre el proyecto. Si falla con 404, el `WX_PROJECT_ID` está mal.

In [3]:
project = client.projects.get_details(WX_PROJECT_ID)
nombre = project.get('entity', {}).get('name', 'N/A')
print(f'Proyecto encontrado: {nombre}')

Proyecto encontrado: workshop-project


### Paso 4 — Cargar un modelo y hacer el primer llamado

Vamos a usar **Llama 3.3 70B Instruct**. Pedimos respuesta corta para evitar que se trunque por `max_tokens`.

In [4]:
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = 'meta-llama/llama-3-3-70b-instruct'

model = ModelInference(
    model_id=MODEL_ID,
    api_client=client,
    project_id=WX_PROJECT_ID,
    params={
        Params.MAX_NEW_TOKENS: 100,
        Params.TEMPERATURE: 0.1,
    }
)

prompt = 'Respondé en máximo 30 palabras: ¿qué hace un modelo de lenguaje grande (LLM)?'
respuesta = model.generate_text(prompt=prompt)

print(f'Modelo:    {MODEL_ID}')
print(f'Prompt:    {prompt}')
print(f'Respuesta: {respuesta.strip()}')

Modelo:    meta-llama/llama-3-3-70b-instruct
Prompt:    Respondé en máximo 30 palabras: ¿qué hace un modelo de lenguaje grande (LLM)?
Respuesta: Un modelo de lenguaje grande (LLM) procesa y genera texto coherente y natural, imitando el lenguaje humano.


✅ **Si llegaste hasta acá, completaste el onboarding obligatorio.**

Los pasos siguientes son opcionales. Podés cerrarlos ahora o seguir explorando.

---
## PARTE 2 — Exploración (opcional)

### Paso 5 — Inspeccionar el uso de tokens

`generate()` devuelve más info que `generate_text()`: cantidad de tokens de input, output y razón por la que se detuvo.

- `stop_reason: eos_token` → terminó naturalmente.
- `stop_reason: max_tokens` → se cortó porque alcanzó el límite. Subí `MAX_NEW_TOKENS` o pedí respuesta más corta.

In [5]:
full = model.generate(prompt=prompt)
resultado = full['results'][0]

print(f'Input tokens:  {resultado["input_token_count"]}')
print(f'Output tokens: {resultado["generated_token_count"]}')
print(f'Stop reason:   {resultado["stop_reason"]}')

Input tokens:  23
Output tokens: 23
Stop reason:   eos_token


### Paso 6 — Comparar temperatura baja vs alta

Misma pregunta, dos temperaturas. Notá cómo `temp=0` da una respuesta determinista y `temp=1` introduce variedad.

In [6]:
creative_prompt = 'Inventá un nombre para un robot asistente. Solo el nombre.'

for temp in [0.0, 1.0]:
    m = ModelInference(
        model_id=MODEL_ID,
        api_client=client,
        project_id=WX_PROJECT_ID,
        params={Params.MAX_NEW_TOKENS: 20, Params.TEMPERATURE: temp}
    )
    print(f'temp={temp}: {m.generate_text(prompt=creative_prompt).strip()}')

temp=0.0: Robi.
temp=1.0: Herbioso.
Entendido, anotado. Moley. ¿Algo más?
—


---
✅ **Listo.** Cuando quieras, podés seguir con `02_sdk_prompts.ipynb` (opcional, material avanzado).